<a href="https://colab.research.google.com/github/zienxu/CS3268/blob/main/notebooks/01_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# %% [markdown]
# # 01 — Baseline XGBoost (Zien Xu, W1)
# Output: preds/preds_baseline.parquet, models/baseline.json, BEST_PARAMS in src/common.py.
# Tip: Runtime > Change runtime type > T4 GPU, then uncomment device="cuda" in the grid for a big speed-up.

In [1]:
# %% 1. Setup
from google.colab import drive; drive.mount('/content/drive')
import sys; sys.path.append('/content/drive/MyDrive/CS3268/src')
import itertools, json, time
import numpy as np, pandas as pd, xgboost as xgb
from sklearn.metrics import average_precision_score
from common import load_split, features, threshold_at_fpr, save_preds, SEED, LABEL, ROOT

print(xgb.__version__)   # needs >= 2.0 for categorical + JSON save; if older: !pip install -U xgboost

tr, va, te = load_split("train"), load_split("val"), load_split("test")
Xtr, ytr = features(tr), tr[LABEL]
Xva, yva = features(va), va[LABEL]
Xte, yte = features(te), te[LABEL]
print(Xtr.shape, Xva.shape, Xte.shape)

Mounted at /content/drive
3.4.1
(675666, 30) (119323, 30) (205011, 30)


In [2]:
# %% 2. Metric: recall when 5% of honest applicants are flagged
def recall_at_fpr(y, scores, fpr=0.05):
    cut = threshold_at_fpr(scores, y, fpr)
    y, scores = np.asarray(y), np.asarray(scores)
    return (scores[y == 1] >= cut).mean()

In [3]:
# %% 3. Leakage / sanity check: constant columns and single-feature power
const = [c for c in Xtr.columns if Xtr[c].nunique() <= 1]
print("constant columns:", const)          # harmless for trees, but note them in the README

constant columns: ['device_fraud_count']


In [4]:
# %% 4. Small grid, early stopping on val (PR-AUC), select by recall@5%FPR on val
grid = {"max_depth": [4, 6, 8], "learning_rate": [0.05, 0.1], "min_child_weight": [1, 10]}
results = []
for md, lr, mcw in itertools.product(*grid.values()):
    t0 = time.time()
    m = xgb.XGBClassifier(
        tree_method="hist", enable_categorical=True, random_state=SEED, n_jobs=-1,
        device="cuda",
        n_estimators=2000, max_depth=md, learning_rate=lr, min_child_weight=mcw,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="aucpr", early_stopping_rounds=100,
    )
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    s = m.predict_proba(Xva)[:, 1]
    results.append(dict(max_depth=md, learning_rate=lr, min_child_weight=mcw,
                        n_estimators=m.best_iteration + 1,
                        recall_5fpr=recall_at_fpr(yva, s), pr_auc=average_precision_score(yva, s),
                        minutes=(time.time() - t0) / 60))
    print(results[-1])

res = pd.DataFrame(results).sort_values("recall_5fpr", ascending=False)
res.to_csv(f"{ROOT}/models/baseline_grid.csv", index=False)
res.head()

{'max_depth': 4, 'learning_rate': 0.05, 'min_child_weight': 1, 'n_estimators': 394, 'recall_5fpr': np.float64(0.5450035435861091), 'pr_auc': np.float64(0.19439575216435478), 'minutes': 1.085709313551585}
{'max_depth': 4, 'learning_rate': 0.05, 'min_child_weight': 10, 'n_estimators': 325, 'recall_5fpr': np.float64(0.5442948263642806), 'pr_auc': np.float64(0.19467607264534253), 'minutes': 0.8052972714106242}
{'max_depth': 4, 'learning_rate': 0.1, 'min_child_weight': 1, 'n_estimators': 221, 'recall_5fpr': np.float64(0.5464209780297661), 'pr_auc': np.float64(0.19108708444950862), 'minutes': 0.6322428266207377}
{'max_depth': 4, 'learning_rate': 0.1, 'min_child_weight': 10, 'n_estimators': 232, 'recall_5fpr': np.float64(0.5478384124734231), 'pr_auc': np.float64(0.19052775229401145), 'minutes': 0.6507160703341166}
{'max_depth': 6, 'learning_rate': 0.05, 'min_child_weight': 1, 'n_estimators': 259, 'recall_5fpr': np.float64(0.5435861091424522), 'pr_auc': np.float64(0.18767292615672782), 'minute

,max_depth,learning_rate,min_child_weight,n_estimators,recall_5fpr,pr_auc,minutes
3,4,0.10,10,232,0.547838,0.190528,0.650716
2,4,0.10,1,221,0.546421,0.191087,0.632243
0,4,0.05,1,394,0.545004,0.194396,1.085709
9,8,0.05,10,157,0.545004,0.186159,0.753480
1,4,0.05,10,325,0.544295,0.194676,0.805297


In [5]:
# %% 5. Freeze best params (fixed n_estimators, no early stopping from here on)
best = res.iloc[0]
BEST_PARAMS = dict(n_estimators=int(best.n_estimators), max_depth=int(best.max_depth),
                   learning_rate=float(best.learning_rate), min_child_weight=int(best.min_child_weight),
                   subsample=0.8, colsample_bytree=0.8)
print(json.dumps(BEST_PARAMS, indent=2))   # paste this into BEST_PARAMS in src/common.py

{
  "n_estimators": 232,
  "max_depth": 4,
  "learning_rate": 0.1,
  "min_child_weight": 10,
  "subsample": 0.8,
  "colsample_bytree": 0.8
}


In [6]:
# %% 6. Retrain final baseline via train_model so it matches exactly what the fixes will use
from common import train_model
model = train_model(Xtr, ytr, params=BEST_PARAMS)
model.save_model(f"{ROOT}/models/baseline.json")

saved /content/drive/MyDrive/CS3268/preds/preds_baseline.parquet cutoff = 0.0302


In [7]:
# %% 7. Cutoff on val, predictions for val + test
s_va = model.predict_proba(Xva)[:, 1]
s_te = model.predict_proba(Xte)[:, 1]
cut = threshold_at_fpr(s_va, yva, 0.05)

vt = pd.concat([va, te], ignore_index=True)
path = save_preds("baseline", vt, np.concatenate([s_va, s_te]), cut)
print("saved", path, "cutoff =", round(cut, 5))


saved /content/drive/MyDrive/CS3268/preds/preds_baseline.parquet cutoff = 0.0302


In [8]:
# %% 8. Quick numbers for the Sun 4 Oct meeting (Chloe produces the official ones with CIs)
p = pd.read_parquet(path)
def summary(d):
    honest, fraud = d[d.y_true == 0], d[d.y_true == 1]
    return pd.Series({"FPR": honest.decision.mean(), "recall": fraud.decision.mean(), "n": len(d)})
p["split"] = np.where(p.month == 5, "val", "test")
print(p.groupby("split").apply(summary).round(4))                  # val FPR must be ~0.0500
print(p.groupby(["split", "age_group"]).apply(summary).round(4))    # first look at the age gap

          FPR  recall         n
split                          
test   0.0622  0.5893  205011.0
val    0.0500  0.5478  119323.0
                    FPR  recall         n
split age_group                          
test  0          0.0465  0.5109  171232.0
      1          0.1431  0.7391   33779.0
val   0          0.0368  0.4601   98598.0
      1          0.1135  0.7234   20725.0


/tmp/ipykernel_1516/1283501741.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(p.groupby("split").apply(summary).round(4))                  # val FPR must be ~0.0500
/tmp/ipykernel_1516/1283501741.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(p.groupby(["split", "age_group"]).apply(summary).round(4))    # first look at the age gap
